# Capstone — Content Refresh: Prioritizing Pages for Editorial Review

This notebook mirrors the deployed paper section by section, and generates the figures
and metrics JSON the paper embeds. Run top to bottom from a fresh clone.

In [ ]:
import os, subprocess, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix,
)

REPO_PATH = "/content/flyrank-ml-internship"
DATA_PATH = f"{REPO_PATH}/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(DATA_PATH):
    if os.path.exists(REPO_PATH):
        subprocess.run(["rm", "-rf", REPO_PATH], check=True)
    subprocess.run(
        ["git", "clone", "https://github.com/muhammadusmanshakir/flyrank-ml-internship.git", REPO_PATH],
        check=True
    )

os.makedirs(f"{REPO_PATH}/work/outputs", exist_ok=True)
os.makedirs(f"{REPO_PATH}/work/figures", exist_ok=True)

print("Environment ready.")

## 1. Question

**Lane:** Content Refresh.

**Question:** which pages should an editorial team prioritize for review when deciding
what to refresh, out of a large and growing backlog?

**Unit of analysis:** one row = one content page.

**Action supported:** an editor opens the top of a ranked queue and reviews flagged
pages against a stated reason code; the model narrows the list, a person decides.

**Cost of a wrong call:** a false positive costs review time on a page that didn't need
it; a false negative delays review of a page that did. Neither is severe on its own,
which is why this stays a decision-support ranking rather than an automated action.

## 2. Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Rows: {len(df):,}  Columns: {len(df.columns)}")

FORBIDDEN = [
    "content_id", "client_id", "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
]

FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "age_tier_order",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

assert not (set(FEATURES) & set(FORBIDDEN)), "Forbidden column found in feature set"
print(f"Feature set: {len(FEATURES)} columns, leakage check passed.")

## 3. Methodology

Target: `decline_target` = 1 when a page's impressions in the most recent 30-day window
are lower than the prior 30-day window. Built only from the two window columns, neither
of which is a model feature. Baseline: `days_since_last_update >= 90 AND
impressions_90d >= 1000` → "Review content", else "Monitor". Model: Random Forest
(`n_estimators=200`, `random_state=42`). Validation: both a random 80/20 stratified
split and a `GroupShuffleSplit` grouped by `client_id`, compared directly.

In [ ]:
df["decline_target"] = (df["impressions_last_30d"] < df["impressions_prev_30d"]).astype(int)

X = df[FEATURES].copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median(numeric_only=True))
y = df["decline_target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

baseline_pred = (
    (X_test["days_since_last_update"] >= 90) & (X_test["impressions_90d"] >= 1000)
).astype(int)

metrics_random = {
    "model_accuracy": accuracy_score(y_test, y_pred),
    "model_f1": f1_score(y_test, y_pred),
    "model_precision": precision_score(y_test, y_pred, zero_division=0),
    "model_recall": recall_score(y_test, y_pred, zero_division=0),
    "model_auc": roc_auc_score(y_test, y_prob),
    "baseline_accuracy": accuracy_score(y_test, baseline_pred),
    "baseline_f1": f1_score(y_test, baseline_pred, zero_division=0),
    "baseline_precision": precision_score(y_test, baseline_pred, zero_division=0),
    "baseline_recall": recall_score(y_test, baseline_pred, zero_division=0),
    "majority_class_rate": float(y_test.value_counts(normalize=True).max()),
}
print(json.dumps(metrics_random, indent=2))

## 4. Results (vs. baseline)

Same target, same test rows, same metrics for both the baseline rule and the model.
Then the honest comparison: same model and features, only the split changes.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])
assert not (train_clients & test_clients), "Client overlap between train and test"

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_grouped.fit(X_train_g, y_train_g)
y_pred_g = model_grouped.predict(X_test_g)
y_prob_g = model_grouped.predict_proba(X_test_g)[:, 1]

metrics_grouped = {
    "grouped_accuracy": accuracy_score(y_test_g, y_pred_g),
    "grouped_f1": f1_score(y_test_g, y_pred_g),
    "grouped_precision": precision_score(y_test_g, y_pred_g, zero_division=0),
    "grouped_recall": recall_score(y_test_g, y_pred_g, zero_division=0),
    "grouped_auc": roc_auc_score(y_test_g, y_prob_g),
    "train_clients": len(train_clients),
    "test_clients": len(test_clients),
}
print(json.dumps(metrics_grouped, indent=2))

with open(f"{REPO_PATH}/work/outputs/capstone_metrics.json", "w") as f:
    json.dump({**metrics_random, **metrics_grouped}, f, indent=2)
print("Saved work/outputs/capstone_metrics.json")

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(10))

fig, ax = plt.subplots(figsize=(8, 5))
importances.head(10).sort_values().plot(kind="barh", ax=ax, color="#0F6B66")
ax.set_title("Top 10 feature importances")
plt.tight_layout()
plt.savefig(f"{REPO_PATH}/work/figures/feature_importance.png", dpi=150)
plt.show()

## 5. Limitations

`decline_target` is a rule-derived proxy for declining visibility, not a human-verified
judgment that a page needs a refresh. The client-grouped validation shows a substantial
generalization gap (ROC-AUC 0.795 → ~0.63 on unseen clients) — real signal, but modest.
The production scoring pass (Section 6) fits on all available rows, so its probability
values are not calibrated confidence and skew high; the ranking is the useful part, not
the raw number. No causal claims are made anywhere in this work.

## 6. Ranked recommendations

In [ ]:
model_final = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model_final.fit(X, y)

playbook_df = df.copy()
playbook_df["decline_probability"] = model_final.predict_proba(X)[:, 1]

def assign_reason(row):
    if row["days_since_last_update"] >= 91:
        return "DECAY_SIGNAL_HIGH"
    elif row["impressions_90d"] < 1000:
        return "LOW_TRAFFIC_SIGNAL"
    elif row["ctr"] < 0.05:
        return "LOW_CTR_SIGNAL"
    else:
        return "REVIEW_REQUIRED"

def assign_action(reason):
    return {
        "DECAY_SIGNAL_HIGH": "Review and refresh stale content",
        "LOW_TRAFFIC_SIGNAL": "Review search visibility and content coverage",
        "LOW_CTR_SIGNAL": "Review title and search-result presentation",
        "REVIEW_REQUIRED": "Perform manual content review",
    }[reason]

playbook_df["reason_code"] = playbook_df.apply(assign_reason, axis=1)
playbook_df["action"] = playbook_df["reason_code"].apply(assign_action)
playbook_df = playbook_df.sort_values("decline_probability", ascending=False).reset_index(drop=True)
playbook_df["rank"] = np.arange(1, len(playbook_df) + 1)

print(playbook_df["reason_code"].value_counts())

action_queue = playbook_df[
    ["rank", "content_id", "action", "reason_code", "decline_probability",
     "days_since_last_update", "impressions_90d", "ctr"]
].copy()
action_queue.to_csv(f"{REPO_PATH}/work/outputs/action_queue.csv", index=False)
print("Saved work/outputs/action_queue.csv")
action_queue.head(10)

## 7. Artifacts the paper embeds

Figures and metrics saved above (`work/figures/feature_importance.png`,
`work/outputs/capstone_metrics.json`, `work/outputs/action_queue.csv`) are the sources
for every number and chart in the deployed paper — nothing in the paper should
contradict what this notebook produces on a fresh run.

In [ ]:
required = ["model", "model_grouped", "metrics_random", "metrics_grouped", "action_queue"]
missing = [r for r in required if r not in globals()]
assert not missing, f"Missing: {missing}"
print("SELF-CHECK PASSED")

## 8. Five-minute demo outline

*For the optional Week-8 showcase.*

**1. Question (30 sec)**
Out of 30,000 pages across 32 client portfolios, which ones should an editorial team
review first for a content refresh — and can we prioritize that list responsibly,
not just by a fixed age rule?

**2. Method (1 min)**
Built a transparent baseline rule first (stale + low-visibility), then a Random Forest
trained on 23 leak-checked features to predict a 30-day decline in impressions. Applied
the same leakage discipline throughout: caught and discarded an earlier version whose
target was identical to the baseline rule (it scored a suspicious 1.0000 accuracy).

**3. One chart (1.5 min)**
Show `charts/honest_validation_drop.png` — the same model, same features, evaluated
two ways: a naive random split vs. a split grouped by client. Every metric drops under
the honest split; ROC-AUC falls from 0.795 to 0.635.

**4. One honest result (1 min)**
That drop *is* the finding worth presenting: the model generalizes to new clients, but
only modestly, not as well as the flattering random-split numbers suggested. Reporting
the weaker, honest number is the actual deliverable of the validation work.

**5. One recommendation (1 min)**
Deploy the ranked queue as decision-support only: an editor works down the list using
the reason codes, nothing is auto-published or auto-deprioritized. Retrain if the
honest ROC-AUC on a fresh client sample drops further below 0.635, or if the
reason-code mix shifts sharply from today's distribution.


## 9. Shareable cuts

**Social post (methodology-focused):**

> Spent the last 7 weeks building a content-refresh prioritization model on 30K
> real search pages across 32 client portfolios. The interesting part wasn't the
> model — it was catching my own mistake: my first version scored a "perfect" 1.0000
> accuracy, which is a red flag, not a win, since the target turned out to be a
> disguised copy of my own baseline rule. Fixed it, then stress-tested the real model
> by validating on clients it had never seen. Accuracy held up reasonably; ROC-AUC
> dropped from 0.80 to 0.64. Reporting that drop, instead of hiding it, was the actual
> point of the exercise. Full writeup + reproducible notebooks linked below.

**Employer-facing summary (3 sentences):**

> I built a machine learning system that ranks content pages by likelihood of
> declining search visibility, trained and evaluated on 30,000 real, anonymized pages
> across 32 client portfolios from FlyRank's search-analytics warehouse. The model
> improved F1 from 0.35 (a hand-written baseline rule) to 0.84 on a standard test
> split, and — critically — I validated it a second, more honest way (holding out
> entire clients rather than random rows), which showed a real but more modest
> generalization gap that I report transparently rather than the more flattering
> number. The result is a reason-coded, human-reviewed action queue, not an automated
> publishing decision, reflecting a decision-support design built around the model's
> actual, honestly-measured limits.
